In [0]:
def process_batch(batch_id):
    """Apply one batch of change message to the hand-build history table."""

    # Step A: If a customer changed twice in one batch, keep only the newest message

    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW latest_changes AS
        SELECT * FROM (
            SELECT *,
                    ROW_NUMBER() OVER (PARTITION BY customer_id
                                ORDER BY change_time DESC) AS rn
            FROM workspace.bank_project.customer_changes
            WHERE batch_id = {batch_id}
        )
        WHERE rn = 1
        """)


    # Step B: close old versions and add new Versions, In one MERGE

    spark.sql("""
          MERGE INTO workspace.bank_project.dim_customer_manual AS t
          USING (
            -- Copy 1: used to find and close and current version
            SELECT s.customer_id AS merge_key, s.*
            FROM latest_changes s

            UNION ALL

            -- Copy 2: used to insert new version for the customers who changed

            SELECT NULL AS merge_key, s.*
            FROM latest_changes s
            JOIN workspace.bank_project.dim_customer_manual d
                ON s.customer_id = d.customer_id AND d.is_current = true
            WHERE s.operation <> 'DELETE'
                AND s.change_time > d.valid_from
                AND NOT (s.full_name <=> d.full_name)
                AND s.city <=> d.city
                AND s.risk_rating <=> d.risk_rating
                AND s.kyc_status <=> d.kyc_status)      
        AS src
        ON t.customer_id = src.merge_key AND t.is_current = true

        WHEN MATCHED
            AND src.change_time > t.valid_from
            AND (src.operation = 'DELETE'
                    OR NOT (src.full_name <=> t.full_name
                        AND src.city  <=> t.city
                        AND src.risk_rating <=> t.risk_rating
                        AND src.kyc_status <=> t.kyc_status))
        THEN UPDATE SET
            t.is_current = false,
            t.valid_to   = src.change_time
                
        WHEN NOT MATCHED AND src.operation <> 'DELETE' THEN
            INSERT (customer_id, full_name, city, risk_rating, kyc_status,
                    valid_from, valid_to, is_current)
                        
            VALUES (src.customer_id, src.full_name, src.city, src.risk_rating,
                    src.kyc_status, src.change_time, NULL, true)
            

              
    """)

    print(f"Batch {batch_id} applied to manual history table.")

def show_history():
    display(spark.sql("""
            SELECT customer_id, full_name, city, risk_rating, kyc_status,
                    valid_from, valid_to, is_current
            FROM workspace.bank_project.dim_customer_manual
            ORDER BY customer_id, valid_from
            
    """))

print("Manual SCD2 tools are ready")

In [0]:
process_batch(1)
show_history()

In [0]:
process_batch(2)
show_history()

In [0]:
def show_auto_history():
    display(spark.sql("""
SELECT customer_id, full_name, city, risk_rating, kyc_status,
__START_AT, __END_AT
FROM workspace.bank_project.customers_scd2_auto
ORDER BY customer_id, __START_AT
"""))
show_auto_history()

In [0]:
process_batch(3)
show_history()

In [0]:
show_auto_history()

In [0]:
point_in_time = "2024-03-01 00:00:00"

print("MANUAL table says:")
display(spark.sql(f"""
    SELECT customer_id, full_name, kyc_status
    FROM workspace.bank_project.dim_customer_manual
    WHERE customer_id = 'C003'
        AND valid_from <= TIMESTAMP '{point_in_time}'
        AND (valid_to > TIMESTAMP '{point_in_time}' OR valid_to IS NULL)
"""))

print("AUTO CDC table says:")
display(spark.sql(f"""
    SELECT customer_id, full_name, kyc_status
    FROM workspace.bank_project.customers_scd2_auto
    WHERE customer_id = 'C003'
        AND __START_AT <= TIMESTAMP '{point_in_time}'
        AND(__END_AT > TIMESTAMP '{point_in_time}' OR __END_AT IS NULL)
"""))

In [0]:
display(spark.sql("""   
    SELECT customer_id, full_name, city, risk_rating, kyc_status
    FROM workspace.bank_project.customers_scd2_auto
    WHERE __START_AT <= TIMESTAMP '2024-04-01 00:00:00'
        AND (__END_AT > TIMESTAMP '2024-04-01 00:00:00' OR __END_AT IS NULL)
    ORDER BY customer_id
"""))

In [0]:
display(spark.sql("""
    SELECT customer_id, full_name, city, risk_rating, kyc_status
    FROM workspace.bank_project.customers_scd2_auto
    WHERE __END_AT IS NULL
    ORDER BY customer_id
"""))

Final check
Test 1: the row counts

In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM workspace.bank_project.customer_changes) AS messages_in_inbox,
    (SELECT COUNT(*) FROM workspace.bank_project.dim_customer_manual) AS manual_rows,
    (SELECT COUNT(*) FROM workspace.bank_project.dim_customer_manual WHERE is_current) AS manual_current,
    (SELECT COUNT(*) FROM workspace.bank_project.customers_scd2_auto) AS auto_rows,
    (SELECT COUNT(*) FROM workspace.bank_project.customers_scd2_auto WHERE __END_AT IS NULL) AS auto_current
"""))   